In [37]:
import mysql.connector
import pandas as pd
from sqlalchemy import create_engine
import os
from pathlib import Path

In [38]:
base_folder = Path(r"/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp")
output_file = base_folder / "combined_skull_analysis.xlsx"

In [39]:
# Defining your SQL query
conn = mysql.connector.connect(option_files=os.path.expanduser('~/.my.cnf')) 

sql_query = """
SELECT o.SPECIMEN_ID, s.STRAIN, s.GENDER, e.DATE, m.COHORT_ID,  o.VALUE_CATEGORY
FROM OBSERVATION as o, MOUSE as m, SPECIMEN as s, EXPERIMENT as e, PARAMETER as p
WHERE o.SPECIMEN_ID=m.ID and m.ID=s.ID and e.ID=o.EXPERIMENT_ID and o.PARAMETER_STABLE_ID=p.PARAMETER_STABLE_ID
 and m.COHORT_ID like 'c%' and o.PARAMETER_STABLE_ID='CCP_XRY_001_001'
"""
sql_query = """
SELECT o.SPECIMEN_ID,s.STRAIN,s.GENDER, m.COHORT_ID, e.DATE,o.VALUE_CATEGORY as SkullShape, LEN.VALUE_DATA_POINT as CraniumLength,COM.VALUE_TEXT as Comment
FROM OBSERVATION as o
    INNER JOIN MOUSE as m ON o.SPECIMEN_ID = m.ID
    INNER JOIN SPECIMEN as s ON m.ID = s.ID
    INNER JOIN EXPERIMENT as e ON e.ID = o.EXPERIMENT_ID
    INNER JOIN PARAMETER as p ON o.PARAMETER_STABLE_ID = p.PARAMETER_STABLE_ID
    LEFT JOIN (
        SELECT SPECIMEN_ID, VALUE_TEXT
        FROM OBSERVATION
        WHERE PARAMETER_STABLE_ID = 'CCP_XRY_032_001'
    ) AS COM ON o.SPECIMEN_ID = COM.SPECIMEN_ID

    LEFT JOIN (
        SELECT SPECIMEN_ID, VALUE_DATA_POINT
        FROM OBSERVATION
        WHERE PARAMETER_STABLE_ID = 'CCP_XRY_893_001'
    ) AS LEN ON o.SPECIMEN_ID = LEN.SPECIMEN_ID
WHERE
    m.COHORT_ID LIKE 'c%'
    AND o.PARAMETER_STABLE_ID = 'CCP_XRY_001_001';
"""
# Reading data into a Pandas DataFrame
db_df = pd.read_sql(sql_query, conn)

# Closing the connection
conn.close()

# Displaying the DataFrame


/usr/local/lib/python3.8/dist-packages/pandas/io/sql.py:761: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


In [40]:
def file_analysis_f(db_cohorts,cohort_df):
    
    res = []
    indexes= []
    for subj in cohort_df.index:
        try:
            specimen_name = subj.split("_")[2]
            
            abn_df = db_cohorts[(db_cohorts['SPECIMEN_ID'] == specimen_name)][['GENDER','STRAIN','COHORT_ID','DATE','CraniumLength','SkullShape','Comment']].values[0].tolist()
            indexes.append(subj )
            #print(cohort_df.loc[subj].values.tolist())

            res.append(abn_df+ cohort_df.loc[subj].values.tolist())
            
        except Exception as e:
            #print(e)
            pass
    return res,indexes

indexes = []
data = []
for fold_ in base_folder.iterdir():
    if fold_.is_file():
        continue
    file_ = (fold_/"abnormality_automated_combined.csv")
    
    if not file_.exists():
        continue
    #print("q")
    file_analysis = pd.read_csv(file_,index_col=0)
    #print(file_analysis)
    td, ind = file_analysis_f(db_df,file_analysis)
    print(td)
    indexes+= ind
    data+=td
    
    
    
    
    
    

[['female', 'Adam7', 'c290', datetime.date(2023, 5, 29), 26.88, 'abnormal', 'cranial sutures open mild', 554.16913348249, 'normal', 0.0156988586141962, 'normal', 1.0263588251187272, 0.9928081110891888, 0.956010137941444], ['male', 'C57Bl/6NCrl', 'c290', datetime.date(2023, 5, 29), 26.69, 'normal', 'kyphosis', 473.2844251317543, 'normal', 0.0088966603896852, 'normal', 1.016555998783327, 0.997176365860613, 0.9675078974964], ['female', 'Cd209c', 'c290', datetime.date(2023, 5, 29), 27.06, 'abnormal', 'T1 not fused, pectus excavatum, cranial sutures open mild', 353.5554759101657, 'normal', 0.0, 'normal', 1.0261306668831185, 1.0027985357576898, 0.9919820237568654], ['female', 'C57Bl/6NCrl', 'c290', datetime.date(2023, 5, 29), 26.87, 'normal', 'T1 not fused, left C7 rib rudiment, pectus excavatum', 450.354122850924, 'normal', 0.0011144844548633, 'normal', 1.0204356556303609, 0.9955758360026, 0.9656873440025074], ['female', 'Adam7', 'c290', datetime.date(2023, 5, 29), 26.39, 'abnormal', 'crani

[['female', 'Trak1 KO', 'c297', datetime.date(2023, 7, 17), 26.82, 'normal', 'T1 not fused', 456.955201276957, 'normal', 0.0007493947196495, 'normal', 1.0101437389513774, 0.9999140444632636, 0.9600148293905852], ['female', 'Mlkl', 'c297', datetime.date(2023, 7, 17), 26.61, 'abnormal', 'cranial sutures open', 457.5100349335903, 'normal', 0.0004419507321009, 'normal', 1.0035055758679603, 1.0003927055780095, 0.9759852003145064], ['male', 'Mlkl', 'c297', datetime.date(2023, 7, 17), 26.77, 'normal', None, 394.6528800566795, 'normal', 0.0, 'normal', 1.0079152557528932, 0.9968233515420772, 0.9839383953999812], ['male', 'Mlkl', 'c297', datetime.date(2023, 7, 17), 28.0, 'normal', 'sternum crooked', 371.7126733547781, 'normal', 0.0, 'normal', 1.0177830589173196, 1.0053979192301064, 0.9900974813553808], ['male', 'Mlkl', 'c297', datetime.date(2023, 7, 17), 26.77, 'normal', None, 433.5658095053968, 'normal', 0.001498789439299, 'normal', 1.01209747457925, 1.0013543949102326, 0.9779786962830576], ['f

[['male', 'Creb3l2', 'c277', datetime.date(2023, 2, 13), 27.08, 'normal', 'T1 not fused', 488.8355298478541, 'normal', 0.0036316821029168, 'normal', 1.0369838101159767, 1.014663418888219, 0.989549677413121], ['male', 'Creb3l2', 'c277', datetime.date(2023, 2, 13), 26.74, 'normal', None, 406.6919888636783, 'normal', 0.0, 'normal', 1.0310862682795696, 0.9920791490445268, 0.9847040149334144], ['female', 'Creb3l2', 'c277', datetime.date(2023, 2, 13), 26.58, 'normal', 'T1 not fused', 309.01296275163685, 'normal', 0.0001921524922178, 'normal', 0.997135413340021, 0.9875787955966888, 0.9600907010008708], ['female', 'Cd209c', 'c277', datetime.date(2023, 2, 13), 26.27, 'normal', None, 412.4462322206223, 'normal', 9.607624610891204e-05, 'normal', 1.0179214674672667, 0.9957466430768188, 0.9776517190535412], ['female', 'Vsx1', 'c277', datetime.date(2023, 2, 13), 28.09, 'abnormal', 'cranial sutures open', 488.1305640392557, 'normal', 0.0, 'normal', 1.0218782483844555, 1.0164765169583996, 0.9890322693

[['female', 'Uox', 'c257', datetime.date(2022, 9, 27), 27.24, 'abnormal', 'T1 extra process, cranial sutures open severe', 602.6868226913023, 'normal', 0.008646862149802, 'normal', 1.0464972767355285, 1.0068038344610415, 0.977951002057268], ['male', 'C57Bl/6NCrl', 'c257', datetime.date(2022, 9, 27), 26.95, 'normal', 'kyphosis', 852.2149473794603, 'normal', 0.0612005687713769, 'normal', 1.0237865164533615, 0.9904644647313228, 0.9896010065200472], ['male', 'Uox', 'c257', datetime.date(2022, 9, 27), 27.27, 'normal', 'pectus excavatum', 463.0874974756133, 'normal', 0.000787825218093, 'normal', 1.01975807402697, 1.0043590504134263, 0.9692194065197022], ['male', 'C57Bl/6NCrl', 'c257', datetime.date(2022, 9, 27), 26.36, 'normal', None, 561.8335526816131, 'normal', 0.0066100457322931, 'normal', 1.02206130933515, 1.0006641202642923, 0.9697191421692396], ['female', 'Uox', 'c257', datetime.date(2022, 9, 27), 26.96, 'normal', 'T1 not fused', 408.3299514919918, 'normal', 0.0006341032243188, 'normal

[['female', 'C57Bl/6NCrl', 'c294', datetime.date(2023, 6, 26), 26.38, 'normal', 'scoliotic mild', 359.8624054112446, 'normal', 0.0002113677414396, 'normal', 1.001081917283106, 0.9819665510508606, 0.9593090516784196], ['male', 'Ism1', 'c294', datetime.date(2023, 6, 26), 24.14, 'abnormal', 'kyphosis, L3 + L4 fused together, shorter rostrum + rostrum malformation + cranial sutures open', 3625.5491729400783, 'abnormal', 0.4524999039237539, 'abnormal', 1.0331714934081917, 0.9985673737015766, 0.9347448000510372], ['male', 'C57Bl/6NCrl', 'c294', datetime.date(2023, 6, 26), 26.7, 'normal', None, 327.8942117945616, 'normal', 0.0002497982398831, 'normal', 1.017645446086858, 0.9854763342629828, 0.9732324265883469], ['male', 'C57Bl/6NCrl', 'c294', datetime.date(2023, 6, 26), 26.59, 'normal', 'scoliotic, kyphosis mild', 641.3133170259279, 'normal', 0.0035356058568079, 'normal', 1.0318667137779376, 0.9941172598626684, 0.9804018537093148], ['male', 'C57Bl/6NCrl', 'c294', datetime.date(2023, 6, 26), 2

...

[['female', 'C57Bl/6NCrl', 'c309', datetime.date(2023, 10, 9), 25.86, 'normal', 'scoliotic, T1 not fused,', 519.2599906861775, 'normal', 0.0254794204680834, 'normal', 0.9994695652920697, 0.9803239362434504, 0.9376557679224704], ['female', 'C57Bl/6NCrl', 'c309', datetime.date(2023, 10, 9), 26.36, 'normal', 'pectus excavatum, kyphosis', 361.47086598011816, 'normal', 0.0, 'normal', 0.9932885398657224, 0.9857675972030876, 0.9614100814105984], ['male', 'Wnt3a', 'c309', datetime.date(2023, 10, 9), 27.48, 'normal', 'T1-T2 transposed, left + right C7 rib rudiment, L6 = half S0, pectus excavatum mild, kyphosis', 593.3603429245338, 'normal', 0.0072057184581684, 'normal', 1.0317265713053625, 0.997120356081898, 0.9882246500072416], ['male', 'Wnt3a', 'c309', datetime.date(2023, 10, 9), 26.93, 'normal', 'Processus spinosus missing, T1 not fused + extra process, left C7 rib rudiment', 774.2061654862852, 'normal', 0.0235386802966834, 'normal', 1.0260990338779652, 0.9673435010521582, 0.9442019974949926

[['male', 'C57Bl/6NCrl', 'c301', datetime.date(2023, 8, 14), 27.11, 'normal', 'T1-T2 transposed, left C7 rib rudiment, lordosis', 623.7793601810238, 'normal', 0.015891011106414, 'normal', 1.023192311548032, 1.0126790503208811, 0.966593210782188], ['female', 'Cd52', 'c301', datetime.date(2023, 8, 14), 27.09, 'abnormal', 'T1 not fused, cranial sutures open', 499.3853803001271, 'normal', 0.0, 'normal', 1.0242718680312648, 0.9909373620099388, 0.9638004739390136], ['female', 'Gzmb', 'c301', datetime.date(2023, 8, 14), 26.18, 'abnormal', 'LHp II. Osteoneogenesis mild, cranial sutures open', 610.1978250166, 'normal', 0.0054187002805426, 'normal', 1.0037061040034403, 0.9677165048411024, 0.9484658882707862], ['female', 'Gzmb', 'c301', datetime.date(2023, 8, 14), 26.94, 'abnormal', 'T1 not fused, cranial sutures open', 615.6183237035254, 'normal', 0.0007686099688712, 'normal', 0.9970609237703498, 0.989339322381298, 0.9346817759294572], ['male', 'Gzmb', 'c301', datetime.date(2023, 8, 14), 25.79, 

[['female', 'Enpp1', 'c288', datetime.date(2023, 5, 15), 25.86, 'abnormal', 'fusion of cervical ,hyperosification: thoracical vert., interscapular, tendon, costal cartilages + sternum , vibrissae follicles,snout , ears, digits', 957.1420402734768, 'normal', 0.0104723108258714, 'normal', 1.014287944551277, 0.9779761659775424, 0.9549638374859138], ['male', 'C57Bl/6NCrl', 'c288', datetime.date(2023, 5, 15), 27.7, 'normal', 'pectus excavatum severe', 709.2995758794409, 'normal', 0.0404673148610737, 'normal', 1.0483726672835754, 1.0272081456182325, 0.9938558788583898], ['male', 'Trak1 KO', 'c288', datetime.date(2023, 5, 15), 26.48, 'normal', 'pectus excavatum severe', 685.0133919501335, 'normal', 0.0275162368855924, 'normal', 1.0590324506835844, 1.0354454654539151, 0.9939902095514872], ['female', 'C57Bl/6NCrl', 'c288', datetime.date(2023, 5, 15), 26.94, 'abnormal', 'T1 not fused, cranial sutures open', 690.5666332785868, 'normal', 0.0698282156719572, 'normal', 1.052590600567261, 1.004543296

[['female', 'C57Bl/6NCrl', 'c311', datetime.date(2023, 10, 23), 25.65, 'abnormal', 'kyphosis, mildly scoliotic, rostrum rounded, cranial sutures open mild', 2322.643223775982, 'abnormal', 0.2280657930133353, 'abnormal', 1.047431033871185, 1.0317557087010556, 0.9228074604354292], ['male', 'Sord', 'c311', datetime.date(2023, 10, 23), 26.94, 'normal', 'T1 not fused', 360.746028722889, 'normal', 0.0003074439875485, 'normal', 1.0175146170320015, 0.9897368440187244, 0.9665910185305836], ['male', 'Sord', 'c311', datetime.date(2023, 10, 23), 26.9, 'abnormal', 'pectus excavatum, cranial sutures open', 398.24520927560457, 'normal', 0.0, 'normal', 1.013544543920334, 0.9942393253680204, 0.9623108361477564], ['male', 'C57Bl/6NCrl', 'c311', datetime.date(2023, 10, 23), 27.2, 'normal', 'C4+C5 fused,', 637.8403159138524, 'normal', 0.0484416432881134, 'normal', 1.0490856921588472, 1.010124419500147, 0.9843442065973336], ['male', 'Sord', 'c311', datetime.date(2023, 10, 23), 27.38, 'normal', 'T1 not fuse

[['male', 'Ssu2', 'c313', datetime.date(2023, 11, 6), 26.89, 'normal', 'pectus excavatum', 265.46843060101264, 'normal', 0.000345874485992, 'normal', 1.012125904685709, 1.0037398068693364, 0.9766037335384056], ['female', 'C57Bl/6NCrl', 'c313', datetime.date(2023, 11, 6), 26.7, 'normal', 'T1 not fused', 340.7195783285061, 'normal', 7.686099688712963e-05, 'normal', 1.0284354604055228, 1.002275848791426, 0.96911945643199], ['male', 'C57Bl/6NCrl', 'c313', datetime.date(2023, 11, 6), 25.52, 'normal', 'L6 = S0, pectus excavatum severe,', 543.1074383720097, 'normal', 0.0112793512931862, 'normal', 0.9967047635034144, 0.9720828136698793, 0.9395152253046636], ['male', 'C57Bl/6NCrl', 'c313', datetime.date(2023, 11, 6), 26.77, 'normal', 'T1 not fused, pectus excavatum', 222.32347766756865, 'normal', 0.0, 'normal', 1.012702395890692, 0.9895702691584498, 0.9794308958819588], ['female', 'Ssu2', 'c313', datetime.date(2023, 11, 6), 25.85, 'abnormal', 'T1 not fused, cranial sutures open', 1078.827507755

In [41]:
res_df = pd.DataFrame(data,index=indexes,columns=['GENDER','STRAIN','COHORT_ID','DATE','CraniumLength','SkullShape','Comment'] + list(file_analysis.columns))

In [42]:

res_df.to_excel(output_file)

#res_df.to_csv(output_file)

/usr/local/lib/python3.8/dist-packages/pandas/compat/_optional.py:149: UserWarning: Pandas requires version '1.2.2' or newer of 'xlsxwriter' (version '1.1.2' currently installed).
  warnings.warn(msg, UserWarning)


In [43]:
file_analysis.columns

Index(['proc_dist', 'abn_distance', 'area99perc', 'abn_perc', 's_x', 's_y',
       's_z'],
      dtype='object')

In [44]:
db_df

,SPECIMEN_ID,STRAIN,GENDER,COHORT_ID,DATE,SkullShape,CraniumLength,Comment
0,064-EPJ77,"Gt(ROSA)26Sor(ACTB-Cre,-EGFP)lcs x Nub1",female,c004,2016-11-10,normal,26.58,None
1,064-EPJ81,"Gt(ROSA)26Sor(ACTB-Cre,-EGFP)lcs x Nub1",male,c004,2016-11-10,normal,26.65,None
2,064-EPJ82,"Gt(ROSA)26Sor(ACTB-Cre,-EGFP)lcs x Nub1",male,c004,2016-11-10,normal,26.21,None
3,064-EPJ85,"Gt(ROSA)26Sor(ACTB-Cre,-EGFP)lcs x Nub1",female,c004,2016-11-09,normal,25.98,None
4,064-EPJ86,"Gt(ROSA)26Sor(ACTB-Cre,-EGFP)lcs x Nub1",female,c004,2016-11-09,normal,26.26,None
...,...,...,...,...,...,...,...,...
9592,E83-F2740,Fam161b,male,c137,2020-05-06,normal,26.56,stretched
9593,E83-F2743,Fam161b,female,c137,2020-05-06,normal,26.37,tendon ossification
9594,E83-F2749,Fam161b,female,c136,2020-05-11,normal,26.52,None
9595,E83-F2750,Fam161b,female,c136,2020-05-11,abnormal,25.31,None


In [45]:
res_df

,GENDER,STRAIN,COHORT_ID,DATE,CraniumLength,SkullShape,Comment,proc_dist,abn_distance,area99perc,abn_perc,s_x,s_y,s_z
Adam7_F_E64-CD965,female,Adam7,c290,2023-05-29,26.88,abnormal,cranial sutures open mild,554.169133,normal,0.015699,normal,1.026359,0.992808,0.956010
wt_M_82Z-13214,male,C57Bl/6NCrl,c290,2023-05-29,26.69,normal,kyphosis,473.284425,normal,0.008897,normal,1.016556,0.997176,0.967508
Cd209c_F_E64-CE021,female,Cd209c,c290,2023-05-29,27.06,abnormal,"T1 not fused, pectus excavatum, cranial suture...",353.555476,normal,0.000000,normal,1.026131,1.002799,0.991982
wt_F_82Z-13253,female,C57Bl/6NCrl,c290,2023-05-29,26.87,normal,"T1 not fused, left C7 rib rudiment, pectus exc...",450.354123,normal,0.001114,normal,1.020436,0.995576,0.965687
Adam7_F_E64-CD970,female,Adam7,c290,2023-05-29,26.39,abnormal,cranial sutures open,548.089089,normal,0.000038,normal,1.030566,1.003992,0.986610
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Ssu2_M_E64-CF733,male,Ssu2,c313,2023-11-06,26.88,normal,T1 not fused,497.191865,normal,0.008090,normal,1.066706,1.024746,0.970470
Ssu2_F_E64-CF731,female,Ssu2,c313,2023-11-06,24.96,abnormal,"kyphosis, rostrum malformation, cranial suture...",3126.643586,abnormal,0.310038,abnormal,1.017994,1.001200,0.962265
Ssu2_F_E64-CF724,female,Ssu2,c313,2023-11-06,26.16,abnormal,cranial sutures open,901.040919,normal,0.040717,normal,1.030944,1.009861,0.937736
wt_M_82Z-14771,male,C57Bl/6NCrl,c313,2023-11-06,26.24,normal,pectus excavatum,362.676440,normal,0.000000,normal,1.014111,0.983307,0.965149
